# Assignment 1

Computer Vision — COMP9320

In [ ]:
# Import the libraries used throughout the assignment.
%matplotlib inline
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Set the input and output paths relative to the project or notebook directory.
ASSIGNMENT_DIR = Path.cwd()
if not (ASSIGNMENT_DIR / "data" / "uni.jpeg").is_file():
    ASSIGNMENT_DIR = ASSIGNMENT_DIR / "Assignments" / "Assign_1"
DATA_DIR = ASSIGNMENT_DIR / "data"
OUTPUT_DIR = ASSIGNMENT_DIR / "outputs"

## Task 1: Image Loading and Grayscale Conversion

### 1. Use OpenCV to read the image uni.jpeg.

In [ ]:
# Read the original color image in OpenCV BGR format.
image_path = DATA_DIR / "uni.jpeg"
original_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
if original_bgr is None:
    raise FileNotFoundError(f"Unable to read image: {image_path}")

### 2. Display the original image and report its properties.

#### Spatial dimensions

- **Pixel:** one spatial sample of a digital image; a color pixel contains values for its color channels.
- **Spatial dimensions:** image height (rows) and width (columns), measured in pixels. For a color array, `shape` is `(height, width, channels)`; `shape[:2]` selects only the spatial dimensions.
- **BGR and RGB:** channel orders for blue, green, and red values. OpenCV loads this image in BGR order; Matplotlib expects RGB. Reordering channels preserves the spatial dimensions and displays the intended colors.

In [ ]:
# Display the original image in RGB order and report its height and width.
original_rgb = cv2.cvtColor(original_bgr, cv2.COLOR_BGR2RGB)
height, width = original_bgr.shape[:2]
print(f"Spatial dimensions (height x width): {height} x {width} pixels")

fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(original_rgb)
ax.set_title("Original image — uni.jpeg")
ax.set_xlabel("Column (pixels)")
ax.set_ylabel("Row (pixels)")
plt.tight_layout()
plt.show()

**Observation:** The image has 476 rows and 645 columns, so it is wider than it is tall. Array indexing uses `(row, column)`; the plot uses columns horizontally and rows vertically.

**References — documentation and further reading:**
- [OpenCV: Basic Operations on Images](https://docs.opencv.org/4.x/d3/df2/tutorial_py_basic_ops.html) — pixel values, BGR order, and array shape.
- [Matplotlib: imshow](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html) — RGB display and image coordinates.

#### Number of color channels

A **color channel** is a two-dimensional array containing one color component at every pixel. This image has blue, green, and red channels. OpenCV stores them in BGR order; each pixel therefore contains three values. The third element of `shape` gives the channel count, whereas `ndim` counts array axes.

In [ ]:
# Report the color channel count, channel order, and array structure.
channel_count = original_bgr.shape[2]
print(f"Number of color channels: {channel_count}")
print("OpenCV channel order: BGR (Blue, Green, Red)")
print("Display channel order: RGB (Red, Green, Blue)")
print(f"Image array shape (height, width, channels): {original_bgr.shape}")
print(f"Number of array axes: {original_bgr.ndim}")
for index, name in enumerate(("Blue", "Green", "Red")):
    print(f"Channel {index}: {name}; spatial shape: {original_bgr[:, :, index].shape}")

**Observation:** There are three color channels, each with 476 rows and 645 columns. The channel axis stores color components; it is not an additional spatial dimension.

**Reference — documentation:** [OpenCV: Basic Operations on Images](https://docs.opencv.org/4.x/d3/df2/tutorial_py_basic_ops.html), sections on image properties and splitting channels.

#### Data type and numerical intensity range

- **Data type (`dtype`):** the representation used to store each channel value. `uint8` is an unsigned 8-bit integer: it represents 256 integer levels from 0 to 255.
- **Channel intensity:** the stored value of one color component at a pixel. Zero is the minimum component level; 255 is the maximum. A single channel value is not the overall brightness of a color pixel.
- **Representable range:** the limits allowed by the data type, obtained with `np.iinfo`. **Observed range:** the minimum and maximum actually present in the image, obtained with `min()` and `max()`. These ranges need not be equal.

In [ ]:
# Report the data type, representable limits, and observed channel intensities.
dtype_limits = np.iinfo(original_bgr.dtype)
print(f"Data type: {original_bgr.dtype}")
print(f"Bits per channel value: {dtype_limits.bits}")
print(f"Representable intensity range: [{dtype_limits.min}, {dtype_limits.max}]")
print(f"Representable intensity levels: {int(dtype_limits.max) - int(dtype_limits.min) + 1}")
print(f"Observed intensity range across all channels: [{original_bgr.min()}, {original_bgr.max()}]")
for index, name in enumerate(("Blue", "Green", "Red")):
    channel = original_bgr[:, :, index]
    print(f"Observed {name} channel range: [{channel.min()}, {channel.max()}]")

**Observation:** The image uses `uint8`. The observed range is [0, 255] both overall and in each color channel. Each channel contains at least one minimum and one maximum value; this does not establish that all 256 levels occur or that a completely black or white pixel exists.

**Reference — documentation:** [NumPy: numpy.iinfo](https://numpy.org/doc/stable/reference/generated/numpy.iinfo.html) — integer limits and bit depth.

#### Memory requirements

- **Byte:** 8 bits. A `uint8` channel value occupies one byte (`itemsize`).
- **Array data memory:** the bytes occupied by the decoded pixel values (`nbytes`), excluding Python object metadata and other arrays. For this image: $M = H \times W \times C \times \text{bytes per channel value}$.
- **Units:** 1 KiB = 1,024 bytes; 1 MiB = 1,048,576 bytes; 1 MB = 1,000,000 bytes.
- **JPEG file size:** compressed storage on disk; it differs from the decoded array memory.

In [ ]:
# Calculate decoded image memory and compare it with the compressed JPEG file size.
pixel_count = height * width
bytes_per_value = original_bgr.itemsize
expected_bytes = pixel_count * channel_count * bytes_per_value
image_bytes = original_bgr.nbytes
jpeg_bytes = image_path.stat().st_size
print(f"Number of spatial pixels: {pixel_count:,}")
print(f"Number of stored channel values: {original_bgr.size:,}")
print(f"Bytes per channel value: {bytes_per_value}")
print(f"Bytes per color pixel: {channel_count * bytes_per_value}")
print(f"Memory calculation: {height} x {width} x {channel_count} x {bytes_per_value} = {expected_bytes:,} bytes")
print(f"Decoded array data (nbytes): {image_bytes:,} bytes")
print(f"Decoded array data: {image_bytes / 1024:.2f} KiB; {image_bytes / 1024**2:.4f} MiB; {image_bytes / 10**6:.5f} MB")
print(f"Compressed JPEG file: {jpeg_bytes:,} bytes ({jpeg_bytes / 1024:.2f} KiB)")
print(f"Calculated memory matches nbytes: {expected_bytes == image_bytes}")

**Observation:** The decoded BGR pixel data occupies 921,060 bytes, whereas the JPEG file occupies 67,589 bytes. JPEG stores a compressed representation. The reported array memory excludes the separate RGB display copy, plotting buffers, and Python object overhead.

**References — documentation:** [NumPy: ndarray.nbytes](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.nbytes.html) and [ndarray.itemsize](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.itemsize.html).

### 3. Convert the original image to grayscale.

**Grayscale** represents each pixel with one intensity value instead of three color components. For `uint8`, 0 represents black and 255 white. OpenCV combines the channels using:

$$Y = 0.299R + 0.587G + 0.114B.$$

The unequal weights approximate the different contributions of the color components to perceived brightness; this is not a simple channel average or a physical luminance measurement. `COLOR_BGR2GRAY` matches the input BGR order. Integer conversion quantizes the result, so manually computed floating-point values may differ slightly.

The conversion preserves height and width but discards color information; the original colors cannot generally be recovered from the grayscale image.

In [ ]:
# Convert the original BGR image to one grayscale channel and report its properties.
gray = cv2.cvtColor(original_bgr, cv2.COLOR_BGR2GRAY)
print("Conversion: cv2.COLOR_BGR2GRAY")
print("Weighted intensity: Y = 0.299R + 0.587G + 0.114B")
print(f"Grayscale shape (height, width): {gray.shape}")
print("Number of grayscale channels: 1")
print(f"Number of array axes: {gray.ndim}")
print(f"Data type: {gray.dtype}")
print(f"Observed intensity range: [{gray.min()}, {gray.max()}]")
print(f"Spatial dimensions preserved: {gray.shape == original_bgr.shape[:2]}")
print(f"Grayscale array data: {gray.nbytes:,} bytes")

**Observation:** The grayscale image is a two-dimensional `uint8` array of shape (476, 645), with intensities from 0 to 255. It stores one value per pixel and uses 307,020 bytes of pixel data, one-third of the original BGR array.

**Reference — documentation:** [OpenCV: Color conversions, RGB ↔ GRAY](https://docs.opencv.org/4.x/de/d25/imgproc_color_conversions.html) — conversion formula and channel-order flags.

### 4. Display the grayscale image.

A **colormap** maps stored values to display colors. `cmap="gray"` displays low intensities as dark and high intensities as light. Setting `vmin=0` and `vmax=255` fixes the display scale to the full 8-bit range; it does not modify the image values. `interpolation="nearest"` avoids smoothing between pixels during rendering.

In [ ]:
# Display the grayscale image with a fixed intensity scale and labeled axes.
fig, ax = plt.subplots(figsize=(8, 6))
gray_plot = ax.imshow(gray, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
ax.set_title("Grayscale image — uni.jpeg")
ax.set_xlabel("Column (pixels)")
ax.set_ylabel("Row (pixels)")
fig.colorbar(gray_plot, ax=ax, label="Grayscale intensity (0–255)")
plt.tight_layout()
plt.show()

**Observation:** The grayscale view retains the spatial layout while removing color distinctions. Dark and light areas represent low and high weighted intensities. Different colors can map to similar gray values, so some color-based distinctions may be lost.

**Reference — documentation:** [Matplotlib: imshow](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html) — colormaps, intensity limits, and interpolation.

## Task 2: Edge Detection Using Canny

### 1. Use the grayscale image obtained in Task 1.

An **edge** is a location of a rapid spatial change in image intensity. It may indicate an object boundary, texture, or a lighting change; not every detected edge is an object boundary. **Edge detection** identifies these changes to emphasize image structure.

Reuse `gray` from Task 1 as the single-channel input. No image reload or second grayscale conversion is needed.

In [ ]:
# Reuse the Task 1 grayscale image and validate the input for edge detection.
assert gray.ndim == 2, "Expected a single-channel grayscale image."
assert gray.dtype == np.uint8, "Expected an 8-bit unsigned integer image."
assert gray.shape == original_bgr.shape[:2], "Spatial dimensions do not match."
canny_input = gray
print("Input source: Task 1 grayscale image (gray)")
print(f"Input shape (height, width): {canny_input.shape}")
print("Number of channels: 1")
print(f"Input data type: {canny_input.dtype}")
print(f"Observed intensity range: [{canny_input.min()}, {canny_input.max()}]")
print(f"Same array as Task 1 grayscale image: {canny_input is gray}")

**Observation:** The input is the existing (476, 645) `uint8` grayscale array. `canny_input = gray` creates another reference to the same array, not a copy; this step does not change its values.

**Reference — further reading:** [OpenCV: Canny Edge Detection](https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html) — intensity changes and the Canny pipeline.

### 2. Apply the Canny edge detection algorithm to extract the edges.
,
,**Gaussian smoothing** averages nearby pixels with greater weight near the center to reduce noise before detecting intensity changes. We explicitly apply a 5 × 5 Gaussian kernel with standard deviation $\sigma=1$ pixel.
,
,A **gradient** measures the spatial change in intensity. Canny uses gradients, thins candidate edges, then links them using **hysteresis**: strong responses above the upper threshold seed edges; weaker candidates between the thresholds survive only when connected to strong edges. Thresholds apply to gradient magnitude, not directly to grayscale values.
,
,Use initial thresholds of 50 and 150. `apertureSize=3` selects a 3 × 3 Sobel derivative kernel; `L2gradient=True` uses $\sqrt{G_x^2+G_y^2}$. The threshold choice will be assessed in Steps 5–6. A **binary edge map** contains two labels: 255 for detected edges and 0 for non-edges.

**Parameter glossary:**

| Term / variable | Meaning in this implementation |
| --- | --- |
| Kernel | A small matrix of filter weights applied over a pixel neighborhood. |
| Sobel operator | A discrete derivative filter approximating intensity changes along columns ($G_x$) and rows ($G_y$). It combines differentiation with smoothing in the perpendicular direction. |
| `sobel_aperture` | Our variable for the Sobel kernel side length; passed to Canny as `apertureSize`. A value of 3 means a 3 × 3 neighborhood, not three passes or a camera aperture. |
| `gaussian_kernel` | Width and height of the separate Gaussian smoothing kernel: (5, 5). |
| `gaussian_sigma` | Gaussian standard deviation in pixels; controls the spread of smoothing weights. |
| `use_l2_gradient` | Our Boolean variable passed as `L2gradient`; `True` selects Euclidean gradient magnitude. |
| `BORDER_REFLECT_101` | Extends image values by reflection without repeating the boundary pixel, allowing filtering near image borders. |

**Reference — documentation:** [OpenCV: Sobel Derivatives](https://docs.opencv.org/4.x/d2/d2c/tutorial_sobel_derivatives.html) — derivative kernels and gradient estimation. The Sobel kernel and the separate Gaussian kernel have different roles.

In [ ]:
# Smooth the grayscale input, apply Canny, and report the settings and output.
gaussian_kernel = (5, 5)
gaussian_sigma = 1.0
lower_threshold = 50
upper_threshold = 150
sobel_aperture = 3
use_l2_gradient = True
smoothed_gray = cv2.GaussianBlur(
    canny_input, gaussian_kernel, sigmaX=gaussian_sigma,
    sigmaY=gaussian_sigma, borderType=cv2.BORDER_REFLECT_101
)
edges = cv2.Canny(
    smoothed_gray, lower_threshold, upper_threshold,
    apertureSize=sobel_aperture, L2gradient=use_l2_gradient
)
edge_pixel_count = np.count_nonzero(edges)
print(f"Gaussian kernel: {gaussian_kernel}; sigma: {gaussian_sigma} pixel")
print("Gaussian border handling: BORDER_REFLECT_101")
print(f"Hysteresis thresholds (lower, upper): ({lower_threshold}, {upper_threshold})")
print(f"Sobel aperture size: {sobel_aperture}; L2gradient: {use_l2_gradient}")
print(f"Edge map shape: {edges.shape}; data type: {edges.dtype}")
print(f"Unique output values: {np.unique(edges).tolist()}")
print(f"Detected edge pixels: {edge_pixel_count:,} / {edges.size:,}")
print(f"Edge pixel percentage: {100 * edge_pixel_count / edges.size:.2f}%")

**Interpretation:** White pixels label retained edges; black pixels label non-edges. The edge pixel percentage describes output density, not detection accuracy. The same smoothing and gradient settings must be retained when comparing thresholds.

**References — documentation and further reading:** [OpenCV: Canny Edge Detection](https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html) and [Canny API](https://docs.opencv.org/4.x/dd/d1a/group__imgproc__feature.html) — algorithm and parameters.

### 3. Display the grayscale image and the resulting edge image side by side.

The **grayscale image** stores intensity values, whereas the **edge map** stores binary edge labels. Both panels use the same spatial coordinates and a fixed 0–255 display range. The left panel shows the original Task 1 grayscale image; Canny was applied to its Gaussian-smoothed version in Step 2.

In [ ]:
# Compare the original grayscale image with the Canny edge map at matching scales.
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)
axes[0].imshow(canny_input, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
axes[0].set_title("Original grayscale image")
axes[1].imshow(edges, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
axes[1].set_title(f"Canny edges — thresholds {lower_threshold}, {upper_threshold}")
for ax in axes:
    ax.set_xlabel("Column (pixels)")
    ax.set_ylabel("Row (pixels)")
fig.suptitle("Grayscale and Canny edge comparison")
plt.tight_layout()
plt.show()

**Observation:** The edge map highlights building outlines, paths, and fine vegetation or architectural texture while discarding the grayscale shading. Some visible responses arise from texture and shadows rather than object boundaries. White pixels represent detected edges, not brighter scene regions; visual comparison alone does not establish detection accuracy.

**Reference — further reading:** [OpenCV: Canny Edge Detection](https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html) — grayscale and edge-map comparison.

### 4. Explain the five fundamental stages of the Canny algorithm.

#### 4.1 Gaussian smoothing

**Smoothing** reduces rapid local intensity variations. A Gaussian filter replaces each pixel with a weighted average of its neighborhood, assigning larger weights near the center:

$$K(i,j)=\frac{\exp\left[-(i^2+j^2)/(2\sigma^2)\right]}{\sum_{u,v}\exp\left[-(u^2+v^2)/(2\sigma^2)\right]},\qquad I_s(x,y)=\sum_{i,j}K(i,j)I(x-i,y-j).$$

Here, $I$ is the grayscale image, $I_s$ is the smoothed image, and $(i,j)$ are kernel offsets. **Normalization** makes the finite kernel weights sum to 1, preserving constant intensities away from numerical rounding.

**Why before differentiation?** Derivatives respond strongly to rapid variations, including noise. Smoothing suppresses some unwanted responses, but also softens real edges and fine texture. Larger $\sigma$ generally increases this tradeoff when the kernel support is sufficient.

**Connection to Step 2:** `cv2.GaussianBlur` explicitly produces `smoothed_gray` with a 5 × 5 kernel and $\sigma_x=\sigma_y=1$. The separate Gaussian operation is not an additional Gaussian stage automatically applied by `cv2.Canny`. Sobel's built-in perpendicular smoothing does not replace this explicit preprocessing.

**References — documentation:** [OpenCV: Smoothing Images](https://docs.opencv.org/4.x/d4/d13/tutorial_py_filtering.html) — Gaussian kernels and sigma; [OpenCV: Canny Edge Detection](https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html) — smoothing before gradient computation.

#### 4.2 Gradient computation

A **partial derivative** measures change along one coordinate while holding the other fixed. The image **gradient** is the vector $\nabla I_s=(G_x,G_y)$: $G_x$ approximates change along columns (left to right), and $G_y$ along rows (top to bottom). Sobel estimates these derivatives from neighboring pixels of the smoothed image.

For the 3 × 3 Sobel operator, the correlation kernels are:

$$K_x=\begin{bmatrix}-1&0&1\\-2&0&2\\-1&0&1\end{bmatrix},\qquad K_y=\begin{bmatrix}-1&-2&-1\\0&0&0\\1&2&1\end{bmatrix}.$$

**Correlation** multiplies each kernel weight by the corresponding neighborhood value and sums the products without flipping the kernel. Thus $G_x$ is positive for intensity increasing to the right, while $G_y$ is positive for intensity increasing downward. A large absolute response indicates a strong change; $G_x$ emphasizes vertical boundaries and $G_y$ horizontal boundaries.

**Code parameters:** `dx` and `dy` are derivative orders: `(1, 0)` computes the first x derivative, and `(0, 1)` the first y derivative. `ksize=sobel_aperture` selects the 3 × 3 kernel. `ddepth=cv2.CV_32F` stores signed 32-bit floating-point responses, preserving negative derivatives that `uint8` cannot represent. `BORDER_REPLICATE` extends the nearest boundary value for derivative filtering.

The following cell illustrates this stage separately; it does not replace the existing `cv2.Canny` result or claim to expose its internal buffers.

In [ ]:
# Compute signed Sobel derivatives of the Gaussian-smoothed image for illustration.
gradient_x = cv2.Sobel(
    smoothed_gray, cv2.CV_32F, dx=1, dy=0,
    ksize=sobel_aperture, borderType=cv2.BORDER_REPLICATE
)
gradient_y = cv2.Sobel(
    smoothed_gray, cv2.CV_32F, dx=0, dy=1,
    ksize=sobel_aperture, borderType=cv2.BORDER_REPLICATE
)
print(f"Sobel kernel size: {sobel_aperture} x {sobel_aperture}")
print("Derivative border handling: BORDER_REPLICATE")
for name, derivative in (("Gx: column-direction change", gradient_x),
                         ("Gy: row-direction change", gradient_y)):
    print(f"{name}; shape: {derivative.shape}; dtype: {derivative.dtype}")
    print(f"  Signed response range: [{derivative.min():.1f}, {derivative.max():.1f}]")

**Interpretation:** Derivative responses are signed changes, not grayscale intensities; their range is not restricted to 0–255. Negative responses indicate decreasing intensity in the corresponding coordinate direction. The next stage combines these components into magnitude and orientation.

**Reference — documentation:** [OpenCV: Sobel Derivatives](https://docs.opencv.org/4.x/d2/d2c/tutorial_sobel_derivatives.html) — derivative kernels, gradient components, and output depth.

#### 4.3 Gradient magnitude and orientation estimation

**Magnitude** is the length of the gradient vector and measures the strength of local intensity change. **Orientation** is the direction of greatest increase in intensity, perpendicular to the local edge tangent:

$$M=\sqrt{G_x^2+G_y^2},\qquad \theta=\operatorname{atan2}(G_y,G_x).$$

`np.hypot(Gx, Gy)` computes the Euclidean magnitude element by element. `np.arctan2(Gy, Gx)` uses both signs to select the correct quadrant and avoids explicit division by zero. Its argument order is y first, x second. It returns **radians** (a full turn is $2\pi$); `np.degrees` converts to **degrees** (a full turn is 360°).

Since image rows increase downward, 0° points right, +90° down, and −90° up. Where magnitude is zero, orientation is undefined; the code marks it as `NaN` (not a number). **Modulo 180°** maps opposite gradient directions to the same unoriented axis, useful for comparing neighbors during non-maximum suppression. These illustrative arrays do not expose Canny's internal buffers.

In [ ]:
# Calculate gradient strength and direction while excluding zero-gradient orientations.
gradient_magnitude = np.hypot(gradient_x, gradient_y)
orientation_degrees = np.degrees(np.arctan2(gradient_y, gradient_x))
valid_orientation = gradient_magnitude > 0
orientation_degrees = np.where(valid_orientation, orientation_degrees, np.nan)
orientation_axis = np.mod(orientation_degrees, 180.0)
print(f"Magnitude shape: {gradient_magnitude.shape}; dtype: {gradient_magnitude.dtype}")
print(f"Magnitude range: [{gradient_magnitude.min():.2f}, {gradient_magnitude.max():.2f}]")
print("Directed angle convention: degrees from +x; positive toward increasing rows")
print("Directed angle interval: [-180, 180] degrees")
print("Unoriented gradient axis interval: [0, 180) degrees")
print(f"Pixels with defined orientation: {np.count_nonzero(valid_orientation):,}")
print(f"Zero-gradient pixels (undefined orientation): {np.count_nonzero(~valid_orientation):,}")
if np.any(valid_orientation):
    print(f"Observed directed angle range: [{np.nanmin(orientation_degrees):.2f}, {np.nanmax(orientation_degrees):.2f}] degrees")

**Interpretation:** Magnitude is nonnegative and can exceed 255 because it is derived from signed Sobel responses, not stored grayscale intensity. A large magnitude alone does not establish an edge: non-maximum suppression and hysteresis still determine which responses survive. At very small magnitudes, orientation is sensitive to small intensity changes.

**References — documentation:** [NumPy: hypot](https://numpy.org/doc/stable/reference/generated/numpy.hypot.html), [NumPy: arctan2](https://numpy.org/doc/stable/reference/generated/numpy.arctan2.html), and [OpenCV: Canny Edge Detection](https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html).